# TP Regularizacion #
Voy a continuar a a partir de del tp de softmax para hacerlo más prolijo ya que no estoy muy conforme con lo realizado en el tp anterior.

#Enunciado
En base a las recomendaciones vistas durante las clases de regularización incluir como hiperparámetros posibles en el trabajo práctico de Clasificación de TNG los siguientes elementos:



Regularización L1/L2, L1+L2
Dropout
BatchNormalization


Para cada caso tener en cuenta las recomendaciones de clase. Mirar valores absolutos de pesos. Tiempos de entrenamiento. Valores de Learning Rate. Ruido en las curvas de aprendizaje. Justificar con lo visto en clase los fenómenos observados.



# TO-DO's

- [x] Implementar normalización con scikit-learn
- [ ] Implementar capas ocultas
- [ ] Implementar Adam con Cyclical Learning Rate
- [ ] Implementar regularización L1 y L2
- [ ] Implementar Dropout
- [ ] Implementar Batch Normalization

## 0. Setup

In [31]:
!pip install -q mlflow optuna


In [32]:
# @title
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


Usando dispositivo: cuda


In [33]:
# @title
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Datos: 20 Newsgroups

Usamos `remove=('headers', 'footers', 'quotes')` para sacar metadata que haría el problema "demasiado fácil" (por ejemplo, el header a veces incluye el nombre del newsgroup).

In [34]:
# @title
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))
#cambie levemente la forma en que se presentan los documentos en el pre preprocesamiento.
#Podemos ver de antemano que hay clases con nombres y contenidos similares, lo que traerá problemas a la hora de clasificar.

class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"\nTrain: {len(newsgroups_train.data)} documentos | Test: {len(newsgroups_test.data)} documentos")
print("\n--- Ejemplo de documento ---")
print("\n".join(f"{i}: {class_names[i]}" for i in range(len(class_names))))



Train: 11314 documentos | Test: 7532 documentos

--- Ejemplo de documento ---
0: alt.atheism
1: comp.graphics
2: comp.os.ms-windows.misc
3: comp.sys.ibm.pc.hardware
4: comp.sys.mac.hardware
5: comp.windows.x
6: misc.forsale
7: rec.autos
8: rec.motorcycles
9: rec.sport.baseball
10: rec.sport.hockey
11: sci.crypt
12: sci.electronics
13: sci.med
14: sci.space
15: soc.religion.christian
16: talk.politics.guns
17: talk.politics.mideast
18: talk.politics.misc
19: talk.religion.misc


In [35]:
# @title
# # @title
# # 1.1 Exploratory Data Analysis - Distribución de clases
# #decidí visualisar la distribucón de clases primero, para conocer bien el dataset.

# train_counts = np.bincount(newsgroups_train.target)
# test_counts  = np.bincount(newsgroups_test.target)

# train_distribution = train_counts / len(newsgroups_train.data)
# test_distribution  = test_counts  / len(newsgroups_test.data)

# print(f"Total de documentos - Train: {len(newsgroups_train.data)}")
# print(f"Media de documentos por clase - Train: {np.mean(train_counts):.2f}")

# print(f"\nTotal de documentos - Test: {len(newsgroups_test.data)}")
# print(f"Media de documentos por clase - Test: {np.mean(test_counts):.2f}")

# plt.figure(figsize=(9, 9))
# plt.pie(train_distribution, labels=class_names, autopct="%1.1f%%")
# plt.title("Distribución de clases - Train")
# plt.show()

# plt.figure(figsize=(9, 9))
# plt.pie(test_distribution, labels=class_names, autopct="%1.1f%%")
# plt.title("Distribución de clases - Test")
# plt.show()

## 2. Preprocesamiento configurable Optuna
Vamos a armar una función de preprocesamiento con estos hiperparámetros (todos van a formar parte de la búsqueda de Optuna más adelante):

- `remove_stopwords` (`True`): Lo vamos a dejar siempre como True
- `normalization` ( `"stem"` / `"lemmatize"`):
  - `"none"`: no se va a considerar este hiperparametro por que no mejora el resultado.

In [36]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text, normalization):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]


    # SIEMPRE SE ELIMINAN STOPWORDS
    #if 1 :   //
    tokens = [sw for sw in tokens if sw not in STOP_WORDS]

    # TODO: aplicar la normalización correspondiente según el valor de `normalization`
    if normalization == "stem":
        tokens = [STEMMER.stem(st) for st in tokens]
        #lt = token stemizado (iterador)
    elif normalization == "lemmatize":
        tokens = [LEMMATIZER.lemmatize(lemT) for lemT in tokens]
    return " ".join(tokens)


In [37]:
# Sanity check: siempre se eliminan stopwords.
ejemplo = (
    "The researchers were studying the running processes "
    "and stopping conditions of the engines."
)

for norm in ["stem", "lemmatize"]:
    resultado = preprocess_text(ejemplo, normalization=norm)
    print(f"normalization={norm:10s} -> {resultado}")

normalization=stem       -> research studi run process stop condit engin
normalization=lemmatize  -> researcher studying running process stopping condition engine


## 3. Vectorización TF-IDF

Convertimos los textos preprocesados en vectores numéricos con `TfidfVectorizer`. Como el preprocesamiento (tokenización, stopwords, normalización) ya lo hicimos nosotros, acá solo controlamos:

- `ngram_range`: `(1,1)` solo unigramas, o `(1,2)` unigramas + bigramas (captura algo de contexto, ej. `"new york"`).
- `max_features`: tamaño máximo del vocabulario (se queda con los términos más frecuentes).
- `min_df`: un término tiene que aparecer en al menos `min_df` documentos para entrar al vocabulario (filtra términos rarísimos / errores de tipeo).

### TODO 2: completar `vectorize_data`

In [38]:
def vectorize_data(train_texts, test_texts, ngram_range, max_features, min_df):
    # TODO: crear un TfidfVectorizer con los parámetros recibidos y ajustarlo (fit) sobre train_texts,
    # después transformar train_texts y test_texts con ese mismo vectorizer.
    my_vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
        min_df=min_df
    )
    X_train = my_vectorizer.fit_transform(train_texts)
    X_test =  my_vectorizer.transform(test_texts)
    return X_train, X_test, my_vectorizer

In [39]:
# Sanity check rápido con pocos documentos
_sample_train = ["the cat sat on the mat", "dogs are running in the park", "cats and dogs are pets"]
_sample_test = ["the dog sat on the mat"]
_Xtr, _Xte, _vec = vectorize_data(_sample_train, _sample_test, ngram_range=(1, 1), max_features=50, min_df=1)
assert _Xtr is not None, "Completá vectorize_data"
print("Shape train:", _Xtr.shape, "| Shape test:", _Xte.shape)
print("Vocabulario (parcial):", list(_vec.vocabulary_.keys())[:10])
feature_names = _vec.get_feature_names_out()
tfidf_values = _Xte.toarray()[0]

for word, score in zip(feature_names, tfidf_values):
    print(f"{word:15s} {score:.4f}")


Shape train: (3, 13) | Shape test: (1, 13)
Vocabulario (parcial): ['the', 'cat', 'sat', 'on', 'mat', 'dogs', 'are', 'running', 'in', 'park']
and             0.0000
are             0.0000
cat             0.0000
cats            0.0000
dogs            0.0000
in              0.0000
mat             0.4338
on              0.4338
park            0.0000
pets            0.0000
running         0.0000
sat             0.4338
the             0.6599


# 3.1 Normalización
Este es el momento más oportuno para normalizar



In [40]:
from sklearn.preprocessing import StandardScaler

def normalize_data(X_train, X_test):
    # Z-score por feature. Se densifica en float32 para reducir el uso de RAM.
    X_train_dense = X_train.toarray().astype(np.float32, copy=False)
    X_test_dense = X_test.toarray().astype(np.float32, copy=False)

    # copy=False permite transformar sobre los mismos arrays cuando es posible.
    scaler = StandardScaler(copy=False)

    X_train_norm = scaler.fit_transform(X_train_dense)
    X_test_norm = scaler.transform(X_test_dense)

    return X_train_norm, X_test_norm, scaler


In [41]:
#sanity check de normalizacion
_Xtr_norm, _Xte_norm, _scaler = normalize_data(_Xtr, _Xte)

print("Shape train normalizado:", _Xtr_norm.shape)
print("Shape test normalizado: ", _Xte_norm.shape)

print("\nMedia por feature en train:")
print(np.round(_Xtr_norm.mean(axis=0), 4))

print("\nDesvío estándar por feature en train:")
print(np.round(_Xtr_norm.std(axis=0), 4))

import pandas as pd

feature_names = _vec.get_feature_names_out()

df_original = pd.DataFrame(
    np.round(_Xtr.toarray(), 3),
    columns=feature_names,
    index=["Doc 1", "Doc 2", "Doc 3"]
)

df_norm = pd.DataFrame(
    np.round(_Xtr_norm, 3),
    columns=feature_names,
    index=["Doc 1", "Doc 2", "Doc 3"]
)

print("=== MATRIZ TF-IDF ORIGINAL ===")
display(df_original)

print("=== MATRIZ NORMALIZADA Z ===")
display(df_norm)

Shape train normalizado: (3, 13)
Shape test normalizado:  (1, 13)

Media por feature en train:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Desvío estándar por feature en train:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
=== MATRIZ TF-IDF ORIGINAL ===


,and,are,cat,cats,dogs,in,mat,on,park,pets,running,sat,the
Doc 1,0.00,0.000,0.398,0.00,0.000,0.00,0.398,0.398,0.00,0.00,0.00,0.398,0.605
Doc 2,0.00,0.349,0.000,0.00,0.349,0.46,0.000,0.000,0.46,0.00,0.46,0.000,0.349
Doc 3,0.49,0.373,0.000,0.49,0.373,0.00,0.000,0.000,0.00,0.49,0.00,0.000,0.000


=== MATRIZ NORMALIZADA Z ===


,and,are,cat,cats,dogs,in,mat,on,park,pets,running,sat,the
Doc 1,-0.707,-1.412,1.414,-0.707,-1.412,-0.707,1.414,1.414,-0.707,-0.707,-0.707,1.414,1.157
Doc 2,-0.707,0.637,-0.707,-0.707,0.637,1.414,-0.707,-0.707,1.414,-0.707,1.414,-0.707,0.126
Doc 3,1.414,0.775,-0.707,1.414,0.775,-0.707,-0.707,-0.707,-0.707,1.414,-0.707,-0.707,-1.283


## 4. Modelo: Regresión Softmax

La regresión softmax (regresión logística multinomial) es el modelo más simple posible para clasificación multiclase: **una sola capa lineal** que mapea el vector de entrada a un score (logit) por clase, sin ninguna no-linealidad ni capa oculta.

`CrossEntropyLoss` de PyTorch ya incluye internamente un `log_softmax` + `NLLLoss`, así que el modelo **no debe aplicar softmax en el forward** — hay que devolver los logits crudos.

### TODO 3: completar `SoftmaxRegression`

In [42]:
class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # TODO: definir una única capa lineal de input_dim a num_classes
        self.linear = nn.Linear(input_dim, num_classes) #implementa (w . X) + b = z, (capa lineal de pytorch)

    def forward(self, x):
        # TODO: devolver los logits (salida de self.linear), SIN aplicar softmax
        return self.linear(x)
#nn.Module
 #  ↓ herencia
#SoftmaxRegression
 #  ↓ contiene
#self.linear
 #  ↓ contiene
#weight y bias

In [43]:
# Sanity check
_dummy = torch.randn(4, 50)
_model_check = SoftmaxRegression(input_dim=50, num_classes=20)
_out = _model_check(_dummy)
assert _out is not None and _out.shape == (4, 20), "Revisá SoftmaxRegression"
print("Shape de salida OK:", _out.shape)


Shape de salida OK: torch.Size([4, 20])


## 5. Entrenamiento (ya resuelto)

Dado que los datos vectorizados entran en memoria sin problema, armamos `DataLoader`s a partir de tensores densos. Logueamos loss y accuracy (train y validación) por época a TensorBoard, y a MLflow además los hiperparámetros y el modelo final.

In [44]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [45]:
def to_tensor_dataset(X, y):
    X_dense = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_dense, y_tensor)


def train_softmax_model(model, train_loader, val_loader, optimizer, epochs, writer=None, run_name="softmax"):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                running_loss += loss.item() * xb.size(0)
                correct += (logits.argmax(dim=1) == yb).sum().item()
                total += xb.size(0)
        val_loss, val_acc = running_loss / total, correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - "
              f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        if writer is not None:
            writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
            writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        # Solo logueamos a MLflow cuando hay un run explícitamente activo.
        # Así Optuna no abre un run implícito que quede activo hasta el entrenamiento final.
        if mlflow.active_run() is not None:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)

    return history


## 6. Búsqueda de hiperparámetros con Optuna

La búsqueda incluye **hiperparámetros de preprocesamiento** (`remove_stopwords`, `normalization`, `ngram_range`, `max_features`, `min_df`) y **de entrenamiento** (`lr`, `batch_size`), todos en un mismo `objective`.

**Optimización de tiempo**: como la tokenización + stopwords + normalización es lo más lento del pipeline (y solo depende de `remove_stopwords` y `normalization`, que tienen pocas combinaciones posibles), la precalculamos una sola vez para las 6 combinaciones posibles y la cacheamos. Así, cada trial de Optuna solo repite la parte rápida (vectorizar + entrenar pocas épocas).

In [30]:
print("Precalculando stemming y lematización...")

preprocessed_cache = {}

for normalization in ["stem", "lemmatize"]:
    train_proc = [
        preprocess_text(text, normalization=normalization)
        for text in newsgroups_train.data
    ]

    test_proc = [
        preprocess_text(text, normalization=normalization)
        for text in newsgroups_test.data
    ]

    preprocessed_cache[normalization] = (train_proc, test_proc)
    print("  listo:", normalization)

Precalculando stemming y lematización...
  listo: stem
  listo: lemmatize


In [ ]:
def objective(trial):
    normalization = trial.suggest_categorical(
        "normalization", ["stem", "lemmatize"]
    )

    ngram_range = trial.suggest_categorical(
        "ngram_range", ["1gram", "1_2gram"]
    )
    ngram_range_tuple = (
        (1, 1) if ngram_range == "1gram" else (1, 2)
    )

    max_features = trial.suggest_categorical(
        "max_features", [2000, 5000, 10000]
    )
    min_df = trial.suggest_categorical("min_df", [1, 2, 5])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128]
    )

    train_texts, _ = preprocessed_cache[normalization]

    tr_texts, val_texts, tr_labels, val_labels = train_test_split(
        train_texts,
        newsgroups_train.target,
        test_size=0.2,
        random_state=SEED,
        stratify=newsgroups_train.target,
    )

    X_tr, X_val, _ = vectorize_data(
        tr_texts,
        val_texts,
        ngram_range_tuple,
        max_features,
        min_df,
    )

    X_tr, X_val, _ = normalize_data(X_tr, X_val)

    train_loader = DataLoader(
        to_tensor_dataset(X_tr, tr_labels),
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        to_tensor_dataset(X_val, val_labels),
        batch_size=batch_size,
        shuffle=False,
    )

    model = SoftmaxRegression(
        input_dim=X_tr.shape[1],
        num_classes=NUM_CLASSES,
    )

    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = train_softmax_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=5,
        run_name=f"trial_{trial.number}",
    )

    return history["val_acc"][-1]


if mlflow.active_run() is not None:
    mlflow.end_run()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor accuracy de validación:", study.best_value)

[I 2026-10-28 18:54:18,404] A new study created in memory with name: no-name-7b53fcc2-9989-4fdf-aa3d-948c89c3c3e0


[trial_0] Epoch 1/5 - train_loss: 8.3492 train_acc: 0.4731 - val_loss: 10.1773 val_acc: 0.5166
[trial_0] Epoch 2/5 - train_loss: 3.6194 train_acc: 0.7325 - val_loss: 12.4768 val_acc: 0.5254
[trial_0] Epoch 3/5 - train_loss: 2.1724 train_acc: 0.8067 - val_loss: 14.2040 val_acc: 0.5139
[trial_0] Epoch 4/5 - train_loss: 1.5902 train_acc: 0.8460 - val_loss: 15.2572 val_acc: 0.5298


[I 2026-10-28 18:54:22,835] Trial 0 finished with value: 0.5201060539107379 and parameters: {'normalization': 'lemmatize', 'ngram_range': '1gram', 'max_features': 2000, 'min_df': 2, 'lr': 0.03125874200890993, 'batch_size': 32}. Best is trial 0 with value: 0.5201060539107379.


[trial_0] Epoch 5/5 - train_loss: 1.0659 train_acc: 0.8784 - val_loss: 16.8064 val_acc: 0.5201
[trial_1] Epoch 1/5 - train_loss: 2.7800 train_acc: 0.5174 - val_loss: 2.9800 val_acc: 0.5718
[trial_1] Epoch 2/5 - train_loss: 0.8767 train_acc: 0.8053 - val_loss: 3.4725 val_acc: 0.5586
[trial_1] Epoch 3/5 - train_loss: 0.4695 train_acc: 0.8757 - val_loss: 3.9463 val_acc: 0.5674
[trial_1] Epoch 4/5 - train_loss: 0.3174 train_acc: 0.9094 - val_loss: 4.4436 val_acc: 0.5625


[I 2026-10-28 18:54:28,574] Trial 1 finished with value: 0.5528060097216085 and parameters: {'normalization': 'stem', 'ngram_range': '1gram', 'max_features': 2000, 'min_df': 1, 'lr': 0.013504960545189707, 'batch_size': 64}. Best is trial 1 with value: 0.5528060097216085.


...

[trial_5] Epoch 5/5 - train_loss: 0.1378 train_acc: 0.9625 - val_loss: 2.4167 val_acc: 0.5798
[trial_6] Epoch 1/5 - train_loss: 1.6868 train_acc: 0.4991 - val_loss: 1.2653 val_acc: 0.6133
[trial_6] Epoch 2/5 - train_loss: 0.5824 train_acc: 0.8357 - val_loss: 1.2944 val_acc: 0.6231
[trial_6] Epoch 3/5 - train_loss: 0.3741 train_acc: 0.9138 - val_loss: 1.3680 val_acc: 0.6111
[trial_6] Epoch 4/5 - train_loss: 0.2774 train_acc: 0.9429 - val_loss: 1.4397 val_acc: 0.6014


[I 2026-10-28 18:54:57,676] Trial 6 finished with value: 0.6005302695536898 and parameters: {'normalization': 'stem', 'ngram_range': '1gram', 'max_features': 2000, 'min_df': 2, 'lr': 0.002662433820410798, 'batch_size': 128}. Best is trial 3 with value: 0.6566504639858595.


[trial_6] Epoch 5/5 - train_loss: 0.2241 train_acc: 0.9566 - val_loss: 1.5049 val_acc: 0.6005
[trial_7] Epoch 1/5 - train_loss: 2.2527 train_acc: 0.5199 - val_loss: 2.0920 val_acc: 0.6465
[trial_7] Epoch 2/5 - train_loss: 0.2711 train_acc: 0.9340 - val_loss: 2.0146 val_acc: 0.6571
[trial_7] Epoch 3/5 - train_loss: 0.1334 train_acc: 0.9673 - val_loss: 1.9993 val_acc: 0.6567
[trial_7] Epoch 4/5 - train_loss: 0.1168 train_acc: 0.9697 - val_loss: 1.9861 val_acc: 0.6611


[I 2026-10-28 18:55:02,559] Trial 7 finished with value: 0.657092355280601 and parameters: {'normalization': 'lemmatize', 'ngram_range': '1gram', 'max_features': 10000, 'min_df': 2, 'lr': 0.0040910845827062, 'batch_size': 128}. Best is trial 7 with value: 0.657092355280601.


[trial_7] Epoch 5/5 - train_loss: 0.1067 train_acc: 0.9718 - val_loss: 1.9671 val_acc: 0.6571


In [ ]:
import optuna.visualization as vis

fig_importance = vis.plot_param_importances(
    study,
    params=[
        "normalization",
        "ngram_range",
        "max_features",
        "min_df",
        "lr",
        "batch_size",
    ],
)

fig_importance.update_layout(
    title="Importancia de los hiperparámetros"
)
fig_importance.show()

fig_history = vis.plot_optimization_history(study)

fig_history.update_layout(
    title="Accuracy de validación por trial"
)
fig_history.show()

print("Mejor accuracy de validación:", study.best_value)
print("\nMejores hiperparámetros:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

## 7. Entrenamiento final

Usamos los mejores hiperparámetros encontrados por Optuna, entrenamos con **todo** el train set y por más épocas, evaluando contra el test set real (que no se usó en la búsqueda).

In [ ]:
best = study.best_params

BEST_NORM = best["normalization"]
BEST_NGRAM = (
    (1, 1) if best["ngram_range"] == "1gram" else (1, 2)
)
BEST_MAX_FEATURES = best["max_features"]
BEST_MIN_DF = best["min_df"]
BEST_LR = best["lr"]
BEST_BATCH_SIZE = best["batch_size"]
EPOCHS_FINAL = 20

train_texts_final, test_texts_final = preprocessed_cache[BEST_NORM]

X_train_final, X_test_final, vectorizer_final = vectorize_data(
    train_texts_final,
    test_texts_final,
    BEST_NGRAM,
    BEST_MAX_FEATURES,
    BEST_MIN_DF,
)

X_train_final, X_test_final, scaler_final = normalize_data(
    X_train_final,
    X_test_final,
)

train_loader_final = DataLoader(
    to_tensor_dataset(X_train_final, newsgroups_train.target),
    batch_size=BEST_BATCH_SIZE,
    shuffle=True,
)

test_loader_final = DataLoader(
    to_tensor_dataset(X_test_final, newsgroups_test.target),
    batch_size=BEST_BATCH_SIZE,
    shuffle=False,
)

model = SoftmaxRegression(
    input_dim=X_train_final.shape[1],
    num_classes=NUM_CLASSES,
)

optimizer = optim.Adam(model.parameters(), lr=BEST_LR)

if mlflow.active_run() is not None:
    mlflow.end_run()

history = train_softmax_model(
    model,
    train_loader_final,
    test_loader_final,
    optimizer,
    epochs=EPOCHS_FINAL,
    writer=None,
    run_name="final",
)

In [ ]:
%tensorboard --logdir runs


## 8. Evaluación final

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader_final:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Accuracy en test: {test_accuracy:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
plt.yticks(range(NUM_CLASSES), class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión - Regresión Softmax sobre 20 Newsgroups")
plt.tight_layout()
plt.show()


In [ ]:
#lista de confusión
TOP_K = 5

top_counts = np.zeros((NUM_CLASSES, TOP_K))
top_labels = np.empty((NUM_CLASSES, TOP_K), dtype=object)

for i in range(NUM_CLASSES):
    fila = cm[i].copy()
    fila[i] = -1  # saco la diagonal: los aciertos no son confusión

    idx = np.argsort(fila)[::-1][:TOP_K]

    for k, j in enumerate(idx):
        top_counts[i, k] = cm[i, j]
        top_labels[i, k] = class_names[j]

plt.figure(figsize=(14, 10))
plt.imshow(top_counts, cmap="Reds", aspect="auto")
plt.colorbar(label="Cantidad de confusiones")

plt.xticks(range(TOP_K), [f"Confusión {i+1}" for i in range(TOP_K)])
plt.yticks(range(NUM_CLASSES), class_names)

for i in range(NUM_CLASSES):
    for j in range(TOP_K):
        plt.text(
            j, i,
            f"{top_labels[i, j]}\n({int(top_counts[i, j])})",
            ha="center",
            va="center",
            fontsize=7
        )

plt.xlabel("Clases con mayor confusión")
plt.ylabel("Clase real")
plt.title("Top 5 confusiones por clase")
plt.tight_layout()
plt.show()

## 9. Preguntas de reflexión

**Sobre Regresión Softmax (teoría)**
1. ¿Por qué la regresión softmax es una generalización de la regresión logística binaria? ¿Qué pasaría si `NUM_CLASSES = 2`?
si k =2
  Basicamente RL_bin es un caso particular de softmax para n=2, matematicamente es lo mismo hacer bxent +sigmoide   que Cxent + softMax, pero si vas a usar 2 clases, te conviene aprovechar la propiedad que  una clase es el complementeo del otro (1-p)  y utilizar la sigmoide.


2. ¿Por qué `CrossEntropyLoss` en PyTorch no necesita que se le aplique `softmax` antes en el forward? ¿Qué pasaría si igual lo aplicaran (softmax + CrossEntropyLoss)?
No termine de entender al 100%, pero se que el código puede ejecutar, pero la función de costo deja de ser la formulación deseada y los gradientes empeoran. Además se pierde estabilidad numérica.


3. ¿Qué tipo de frontera de decisión puede aprender un modelo de regresión softmax? ¿Qué limitación tiene frente a un problema que no es linealmente separable?  Puede aprender un hiperplano de R^n-1, y la limitación que tiene es que si las fronteras no son lineales (ejemplo, caso de distribución en forma de V en pesos y alturas), estos problemas tienen difucultades e interpolan mal, para soluciónar esto se puede agregar capas ocultas.


4. ¿En qué se diferencia este modelo de una red neuronal con capas ocultas? ¿En qué situación elegirían una sobre la otra?Este tiene una solo capa lineal,entonces permite trackear la sensibiliad del modelo respecto de cada parametro. Entonces se puede justificar un resultado, como ocurre en el otorgamiento de creditos bancarios. También estos modelos de una capa de transformación lineal justamente son más simples pero con más riesgo de underfiting (tienden a generalizar más que muchas capas ocultas).

**Sobre frameworks y herramientas**
5. ¿Qué ventajas y desventajas tiene implementar esto en PyTorch (como hicimos) comparado con usar directamente `sklearn.linear_model.LogisticRegression(multi_class="multinomial")`?
  PyTorch da mayor control: puedo definir explícitamente la arquitectura, loss, optimizador, batches, épocas, GPU y logging del entrenamiento. Eso lo hace flexible y extensible hacia modelos más complejos


6. ¿Para qué sirvió específicamente MLflow en este TP? ¿En qué se diferencia de lo que loguea TensorBoard?
En este TP, MLflow sirve para registrar experimentos: hiperparámetros, métricas y el modelo entrenado, permitiendo comparar y reproducir ejecuciones. TensorBoard está principalmente orientado a visualizar la dinámica del entrenamiento, por ejemplo:

7. ¿Qué ventaja concreta les dio Optuna frente a probar combinaciones de hiperparámetros a mano?
evaluó cada configuración usando accuracy de validación.

**Sobre el experimento**

8. ¿Qué combinación de preprocesamiento (stopwords, stemming/lematización, n-gramas) encontró Optuna como mejor? ¿Tiene sentido para este dataset?

Optuna encontró como mejor combinación usar stopwords, stemming, unigramas + bigramas, max_features=5000 y min_df=5. Tiene sentido porque al sacar stopwords se eliminan palabras muy frecuentes que aportan poca información, y el stemming agrupa palabras parecidas reduciendo el vocabulario. Usar unigramas y bigramas también puede ayudar porque no solo mira palabras sueltas sino algunas combinaciones de dos palabras. Igual, esto hay que tomarlo como el mejor resultado dentro de los 15 trials probados, no como una combinación óptima definitiva


9. ¿En qué categorías del dataset se confunde más el modelo (según la matriz de confusión)? ¿Por qué creen que pasa (piensen en el contenido temático de esas categorías)?

El modelo se confunde principalmente entre categorías que tienen contenido temático parecido. Por ejemplo, talk.politics.misc se confunde mucho con talk.politics.guns, soc.religion.christian con talk.religion.misc, y alt.atheism también con categorías relacionadas a religión. En computación ocurre algo similar: comp.graphics, comp.windows.x y comp.os.ms-windows.misc presentan bastante confusión entre sí. También rec.autos se confunde con rec.motorcycles. Esto tiene sentido porque el modelo trabaja sobre las palabras y n-gramas presentes en el texto, entonces categorías cercanas comparten mucho vocabulario y pueden quedar próximas en el espacio de features.


10. ¿Cambiaría su respuesta a la pregunta anterior si usaran un modelo con capas ocultas en vez de regresión softmax? ¿Por qué sí o por qué no?

Sí, podría cambiar parcialmente. Una red con capas ocultas puede aprender relaciones no lineales entre las features, mientras que la regresión softmax solo puede separar las clases mediante fronteras lineales. Esto podría ayudar a distinguir mejor categorías que comparten vocabulario pero donde la combinación de palabras tiene significados distintos. Sin embargo, no necesariamente eliminaría todas las confusiones, porque algunas clases realmente tienen contenido muy parecido y además una red más compleja también puede aumentar el riesgo de overfitting.

11. Analisis de tiempos

Los distintos entrenamientos convergen a velocidades diferentes principalmente por el lr. Los casos con learning rate alto bajan muy rápido el error de entrenamiento, pero tienden a volverse inestables y tienen peor desempeño en validación. En cambio, los mejores trials usaron valores de \(lr\) cercanos a \(10^{-3}\), donde la convergencia fue más suave y estable.
Por ejemplo, el trial 11, que fue el mejor, pasó de una val_acc de 0.6611 en la época 1 a un máximo de 0.6681 en la época 2, y luego se mantuvo alrededor de ese valor. A partir de ahí el train_loss siguió bajando, pero el val_loss empezó a aumentar, indicando que el modelo ya había convergido en términos de generalización y empezaba a sobreajustar.
En cambio, con valores altos de lr, como en el trial 2, el train_loss baja muy rápido pero el val_loss permanece muy alto, por lo que converger rápido sobre train no implica necesariamente una mejor solución.
En conclusión, los mejores entrenamientos convergen en pocas épocas, aproximadamente entre la primera y la segunda época en validación. Seguir entrenando mejora el ajuste sobre train, pero no necesariamente la capacidad de generalización.
